<a href="https://colab.research.google.com/github/wricheek84/deep-learning-from-scratch/blob/main/cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!nvcc --version

Thu Nov 12 07:50:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile device_query.cu
#include <stdio.h>
#include <cuda_runtime.h>

int main() {
    int dc = 0;
    cudaError_t error = cudaGetDeviceCount(&dc);
    if (error != cudaSuccess) {
        fprintf(stderr, "Failed to get device count: %s\n", cudaGetErrorString(error));
        return -1;
    }
    if (dc == 0) {
        printf("No CUDA devices found.\n");
        return 0;
    }
    printf("Number of CUDA devices: %d\n", dc);
    for (int i = 0; i < dc; i++) {
        cudaDeviceProp dp;
        cudaGetDeviceProperties(&dp, i);
        int memClockKHz = 0;
        cudaDeviceGetAttribute(&memClockKHz, cudaDevAttrMemoryClockRate, i);
        double peakBandwidth = 2.0 * (memClockKHz * 1000.0) * (dp.memoryBusWidth / 8.0) / 1e9;

        printf("Device %d: %s\n", i, dp.name);
        printf("compute capability: %d.%d\n", dp.major, dp.minor);
        printf("Total global memory: %.2f GB\n", dp.totalGlobalMem / (1024.0f * 1024.0f * 1024.0f));
        printf("number of sms: %d\n", dp.multiProcessorCount);
        printf("warp size: %d\n", dp.warpSize);
        printf("maximum threads per block: %d\n", dp.maxThreadsPerBlock);
        printf("max threads per sm: %d\n", dp.maxThreadsPerMultiProcessor);
        printf("registers per block: %d\n", dp.regsPerBlock);
        printf("shared memory per block: %zu bytes\n", dp.sharedMemPerBlock);
        printf("maximum grid size: %d x %d x %d\n", dp.maxGridSize[0], dp.maxGridSize[1], dp.maxGridSize[2]);
        printf("maximum block size: %d x %d x %d\n", dp.maxThreadsDim[0], dp.maxThreadsDim[1], dp.maxThreadsDim[2]);
        printf("Peak memory bandwidth: %.2f GB/s\n", peakBandwidth);
    }
    return 0;
}

Writing device_query.cu


In [3]:
!nvcc -arch=sm_75 -O3 device_query.cu -o device_query && ./device_query

Number of CUDA devices: 1
Device 0: Tesla T4
compute capability: 7.5
Total global memory: 14.56 GB
number of sms: 40
warp size: 32
maximum threads per block: 1024
max threads per sm: 1024
registers per block: 65536
shared memory per block: 49152 bytes
maximum grid size: 2147483647 x 65535 x 65535
maximum block size: 1024 x 1024 x 64
Peak memory bandwidth: 320.06 GB/s
